# 1 loading data and dependencies 

In [1]:
import sklearn as sk
import pandas as pd


In [25]:
df = pd.read_csv("data/vehicles.csv")
df["url"]


0         https://prescott.craigslist.org/cto/d/prescott...
1         https://fayar.craigslist.org/ctd/d/bentonville...
2         https://keys.craigslist.org/cto/d/summerland-k...
3         https://worcester.craigslist.org/cto/d/west-br...
4         https://greensboro.craigslist.org/cto/d/trinit...
                                ...                        
426875    https://wyoming.craigslist.org/ctd/d/atlanta-2...
426876    https://wyoming.craigslist.org/ctd/d/atlanta-2...
426877    https://wyoming.craigslist.org/ctd/d/atlanta-2...
426878    https://wyoming.craigslist.org/ctd/d/atlanta-2...
426879    https://wyoming.craigslist.org/ctd/d/atlanta-2...
Name: url, Length: 426880, dtype: str

## Droping useless cols


In [3]:
# colls to drop id, image_url, region_url, posting_date, county
# url, VIN, description, lat, long are kept for the iteration 2 listing fields (built at the end)
RAW_LISTING_COLS = ["url", "VIN", "description", "lat", "long"]
# no in place so runing 2 times does not raise error 
df = df.drop(columns=[
    "id",
    "image_url",
    "region_url",
    "posting_date",
    "county"
])


## Miising price = drop

In [4]:
#deleting all posts that cars have no price 
df = df.drop(df[df["price"] == 0].index)


# Droping classic cars
-they follow diffrent pricing

In [5]:
df = df[df["year"] >= 1980]

# droping region ins in between state

In [6]:

df = df.drop(columns=["region"])


# size = drop 
- it is gonna have colinearity with model

In [7]:
df = df.drop(columns=["size"])

## Price
- droping fake listings
- dropping new cars that have the price finanncing like not full price
- casting float to int 

In [8]:
#deleting all prices that are fake / paceholders 
fake_cheap = (df["price"] < 1_000) & (df["year"] >= 2010)
df = df[(df["price"] >= 500) & ~fake_cheap & (df["price"] <= 200_000)]

df = df[~((df["year"] >= 2021) & (df["price"] < 5_000))]

df["year"] = df["year"].astype(int)

## condition filling

In [9]:
#Condition filling all na
df["condition"] = df["condition"].fillna("unknown")

## paint_color filling + joining fancy ones

In [10]:
# all na are unknown colours then using common colours as a bucket and others fancy as one category
common_cols = ["white", "black", "silver", "blue", "red", "grey"]

df["paint_color"] = (df["paint_color"]
                     .where(df["paint_color"].isin(common_cols) | 
                            df["paint_color"].isna(), "not ordinary")
                            .fillna("missing"))

## brands
- brands are fixed so the all spell the same 
- for missing brands i try to take out the first word in the the model if it is in my allowed brands brand is updatedm

In [11]:
# if the have model name taking brand from model (people headbutted the keyboard so some changes needed)
extra_brands = {"scion", "international", "genesis", "freightliner", "isuzu", "hummer",
                "oldsmobile", "maserati", "saab", "suzuki", "smart", "plymouth",
                "kenworth", "hino", "peterbilt", "sterling", "mack", "mg", "triumph", "bentley"}

miss_spellings = {"chevorlet": "chevrolet", "volkswagon": "volkswagen", "gmc,": "gmc"}

first_word = df["model"].str.split().str[0].str.lower().replace(miss_spellings)
known = set(df["manufacturer"].dropna().unique()) | extra_brands
df["manufacturer"] = (df["manufacturer"]
                      .fillna(first_word.where(first_word.isin(known)))
                      .fillna("unknown"))

## Year col
-its a srong colum so ill just drop missing values less tanh 2k rows

In [12]:
df = df.drop(df[df["year"].isna()].index)

## Model
-tricky one with a lot different ones so i ran an agent to fix and simplify models and save it in `data/fixed_models.csv` (the agent is in `model_names.ipynb`)
- later ill use only if there is more that 50 of that model 

In [13]:
df["model"] = pd.read_csv("data/fixed_models.csv", index_col=0)["model"]

# Odometer 
- drop missing values
- drop fake values like old cars with no milage 
- or where milage is too big a  

In [14]:
df = df.dropna(subset=["odometer"])
df = df[df["odometer"] <= 500_000]
df = df[~((df["odometer"] < 100) & (df["year"] < 2020))]

# Fuel just drop na

In [15]:
df = df.dropna(subset=["fuel"])

# title_status
- strong feature where ill just drop the missing ones 

In [16]:
df = df.dropna(subset=["title_status"])

# Cylinders 
- assingning electric to others 
- assigning 4 cil to hybrids (99% of the cars are)
- search model and assign the most comon in that class
- drop unknow to all others (709)

In [17]:
df.loc[df["cylinders"].isna() & (df["fuel"] == "electric"), "cylinders"] = "other"
df.loc[df["cylinders"].isna() & (df["fuel"] == "hybrid"), "cylinders"] = "4 cylinders"

mode_per_model = df.groupby(["manufacturer", "model"])["cylinders"].transform(
    lambda x: x.mode()[0] if x.notna().any() else None
)
df["cylinders"] = df["cylinders"].fillna(mode_per_model)

df = df.dropna(subset=["cylinders"])


# Transmission its a very smal na (1.4k)
-droping the listings

In [18]:
df = df.dropna(subset=["transmission"])

# Type 
- evaluate by model then assinging a type by model 
- filling the rest as unknown 

In [19]:
mode_per_model = df.groupby(["manufacturer", "model"])["type"].transform(
    lambda s: s.mode()[0] if s.notna().any() else None
)
df["type"] = df["type"].fillna(mode_per_model)
df["type"] = df["type"].fillna("unknown")

# drive (same method as type) 
- evaluate by model then assinging a drive by model 
- filling the rest as unknown 

In [20]:
def confident_mode(s, min_share=0.8):
    counts = s.value_counts(normalize=True)
    if counts.empty or counts.iloc[0] < min_share:
        return None
    return counts.index[0]

mode_per_model = df.groupby(["manufacturer", "model"])["drive"].transform(confident_mode)
df["drive"] = df["drive"].fillna(mode_per_model)
df["drive"] = df["drive"].fillna("unknown")


# finnally droping dupes

In [21]:
# raw listing cols (url, description, ...) differ between reposts, so they're left out of the comparison
df = df.drop_duplicates(subset=[c for c in df.columns if c not in ["state"] + RAW_LISTING_COLS])

# Iteration 2: listing fields
- `is_dealer` from the url (craigslist `ctd` = dealer, `cto` = owner)
- `has_vin` + adding keywords from description `desc_len` `needs_work`, `high_trim`, `lifted`
- (impossible coords → NaN)

The raw text columns are dropped after. Keyword patterns were picked in `EDA.ipynb` (iteration 2); the column names are listed in `src/carpricepredictor/shared.py` (`listing_cols`) so the other notebooks know them.

In [22]:
DESCRIPTION_FLAGS = {
    "needs_work": r"needs? (?:some )?work|mechanic special|\bas[- ]is\b|project car|for parts|not running"
                  r"|doesn.?t run|won.?t start|blown|bad (?:engine|transmission)",
    "high_trim": r"\b(?:platinum|limited|king ranch|denali|lariat|laramie|longhorn|ltz|high country|premier"
                 r"|signature|overland|summit|sahara|rubicon|raptor|trd pro)\b",
    "lifted": r"lifted|lift kit",
    "diesel_words": r"diesel|powerstroke|duramax|cummins",
    "one_owner": r"one owner|1 owner|single owner",
    "carfax": r"carfax",
    "warranty": r"warranty",
    "leather": r"leather",
    "sunroof": r"sunroof|moonroof|panoramic",
    "navigation": r"navigation|\bnav\b",
    "turbo": r"turbo",
}

def listing_fields(df):
    """Raw url / VIN / description / lat / long -> numeric listing features."""
    text = df["description"].fillna("").str.lower()
    out = pd.DataFrame(index=df.index)
    out["is_dealer"] = df["url"].str.contains("/ctd/").astype(int)  # craigslist: ctd = dealer, cto = owner
    out["has_vin"] = df["VIN"].notna().astype(int)
    out["desc_len"] = text.str.len()
    in_us = df["lat"].between(18, 72) & df["long"].between(-170, -65)  # a few listings have impossible coords
    out["lat"] = df["lat"].where(in_us)
    out["long"] = df["long"].where(in_us)
    for name, pattern in DESCRIPTION_FLAGS.items():
        out[name] = text.str.contains(pattern).astype(int)
    return out

df = pd.concat([df.drop(columns=RAW_LISTING_COLS), listing_fields(df)], axis=1)
df.shape

(205385, 30)

# DF TO CSV 

In [24]:
df.to_csv("data/vehicles_clean.csv", index=False)